# Open-weight Phase 4 inference (tb-llm-equity)

Runs the 400 Phase-3 arm prompts through four open-weight models, fully local (vLLM on a Colab T4), zero API cost. See `results/PREREGISTRATION.md` Amendments (2026-08-06, open-weight pivot) for why: zero compute budget authorized, single seed at temperature 0, 4-bit quantization throughout.

**Two phases, in order -- do not skip the gate between them:**

1. **Pilot** (Task A): 20 stratified prompts (5 vignettes x 4 arms) through Llama-3.1-8B-Instruct, then the same 20 through Meditron3-8B. Reports real measured output-token stats, throughput, response parseability, and a corrected full-run ETA -- the planning-time 700-tokens/response assumption was never validated against a real completion, and this is where that happens, before you commit hours to the full run.
2. **Full run** (Task B): all 4 models x 400 prompts x 1 seed. Checkpoints to Drive after **every single response**, not per-batch -- free Colab disconnects without warning, and a lost session should cost you minutes of re-warmup, not hours of regenerated work. Resumable: re-running this notebook after a disconnect skips every response already checkpointed.

**What was and wasn't tested before you run this.** The checkpoint/resume logic (`src/tb_equity/checkpoint.py`) and the response-parseability heuristic (`src/tb_equity/response_format.py`) are unit-tested against a stub generator in `tests/test_checkpoint.py` / `tests/test_response_format.py` -- including a simulated mid-write kill. **The vLLM/model-loading cells below have not been run against a real GPU** -- this environment had no GPU or Colab execution channel available when this notebook was built. A static review (read, not run) caught and fixed several T4-specific issues before this version: an unpinned vLLM version, `bfloat16` vs. `float16` (T4/Turing lacks bf16 tensor cores), `tokenizer_revision` not pinned alongside `revision`, and -- the biggest one -- the full run originally called `LLM.generate()` once per prompt in a loop to get per-response checkpointing, which would have silently forfeited continuous batching entirely and made the pilot's throughput measurement not comparable to the full run's actual speed; it now uses `LLMEngine.add_request()`/`.step()` to get both. None of this substitutes for an actual run -- treat the first full run of each cell as the real first test of it.

## 1. Install dependencies

`vllm` is version-pinned (0.26.0, the latest PyPI release as of 2026-08-06, verified by web search when this notebook was built) rather than left floating -- an unpinned `pip install vllm` picks up whatever is newest at *run* time, which is a real way for a notebook that worked when written to break when actually run.

In [ ]:
!pip install -q vllm==0.26.0 transformers accelerate bitsandbytes huggingface_hub pyyaml

## 2. Mount Drive and locate the repo

Checkpoints always go to Drive (`DRIVE_ROOT`), regardless of where the repo itself comes from, so they survive a disconnect even if the repo was git-cloned into ephemeral `/content` storage.

Set exactly one of `REPO_SOURCE` (a git remote URL, if you've pushed this repo somewhere) or sync the repo folder into `DRIVE_ROOT` yourself before running this cell -- this repo had no git remote configured when this notebook was built (`git remote -v` was empty), so a Drive sync is the default path.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/tb-llm-equity')  # adjust if you synced elsewhere
REPO_SOURCE = None  # e.g. 'https://github.com/you/tb-llm-equity.git' -- leave None to use DRIVE_ROOT directly

if REPO_SOURCE:
    import subprocess
    subprocess.run(['git', 'clone', REPO_SOURCE, '/content/repo'], check=True)
    REPO_DIR = Path('/content/repo')
else:
    REPO_DIR = DRIVE_ROOT

assert REPO_DIR.exists(), (
    f'{REPO_DIR} not found -- sync the repo into your Drive at this path, '
    'or set REPO_SOURCE to a git URL above.'
)

PROMPTS_DIR = REPO_DIR / 'data' / 'prompts'
MODELS_CONFIG_PATH = REPO_DIR / 'config' / 'models.yaml'
VIGNETTES_DIR = REPO_DIR / 'data' / 'vignettes' / 'v1'
assert PROMPTS_DIR.exists(), f'{PROMPTS_DIR} not found -- run scripts/expand_arms.py in the main repo first.'

# Checkpoints, final responses, and pilot results ALWAYS go to Drive directly,
# independent of REPO_SOURCE -- created now, before any generation happens below,
# so the first checkpoint/pilot-result write always lands on durable storage.
RESPONSES_DIR = DRIVE_ROOT / 'data' / 'responses'
MANIFEST_DIR = DRIVE_ROOT / 'results' / 'manifests'
PILOT_RESULTS_DIR = DRIVE_ROOT / 'results' / 'pilot'
RESPONSES_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
PILOT_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 3. Import tb_equity (checkpoint/resume + parseability + cache addressing)

Reuses the exact, unit-tested modules from the main repo rather than duplicating their logic in notebook cells -- `cache_key_for` also makes the checkpoint filenames byte-identical to what `CachingLLMClient` would have produced for the same (model, prompt, params), so the Phase 5 scorer never needs to know which path -- API or this notebook -- produced a given file.

In [ ]:
import sys
sys.path.insert(0, str(REPO_DIR / 'src'))

import yaml

from tb_equity.checkpoint import (
    checkpoint_path,
    load_completed_keys,
    pending_keys,
    write_response_atomic,
)
from tb_equity.llm_client import GenerationParams, cache_key_for
from tb_equity.response_format import check_parseable

## 4. HuggingFace login

Two of the four models need a token that has **accepted the model's license** on huggingface.co before download will work:

- `meta-llama/Llama-3.1-8B-Instruct` -- manually gated (visit the model page, request access, wait for approval). The AWQ checkpoint we actually load (`hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4`) is itself ungated, but vLLM still needs the base tokenizer/config which may reference the gated repo.
- `EPFLiGHT/Meditron3-8B` -- auto-gated (accept terms on the model page, access is granted immediately).

Store your token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar), not hardcoded here.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

## 5. Model registry (from config/models.yaml -- single source of truth)

In [ ]:
with open(MODELS_CONFIG_PATH, encoding='utf-8') as f:
    models_cfg = yaml.safe_load(f)

MODEL_REGISTRY = [m for m in models_cfg['evaluation']['models'] if m.get('runtime') == 'colab_vllm']
assert len(MODEL_REGISTRY) == 4, f'expected 4 open-weight models, found {len(MODEL_REGISTRY)}'

for m in MODEL_REGISTRY:
    print(f"{m['family']:8s} {m['model']:55s} rev={m['revision'][:12]} quant={m['quantization']}")

## 6. Load the 400 prompts and build the stratified 20-prompt pilot sample

5 vignettes (one per `presentation_type`, deterministically the lowest `VIG-id` in each stratum) x 4 arms = 20 prompts, per Task A.

In [ ]:
import json

prompts_manifest = json.loads((PROMPTS_DIR / 'manifest.json').read_text(encoding='utf-8'))
all_entries = prompts_manifest['prompts']  # 400: vignette_id, arm, path, sha256, estimated_tokens, burden_class
assert len(all_entries) == 400

vignette_meta = {}
for p in VIGNETTES_DIR.glob('VIG-*.json'):
    v = json.loads(p.read_text(encoding='utf-8'))
    vignette_meta[v['id']] = v

by_type: dict[str, list[str]] = {}
for vid, v in vignette_meta.items():
    by_type.setdefault(v['presentation_type'], []).append(vid)

PILOT_VIGNETTE_IDS = sorted(min(ids) for ids in by_type.values())  # deterministic, 1 per type
print('Pilot vignettes (1 per presentation_type):', PILOT_VIGNETTE_IDS)
assert len(PILOT_VIGNETTE_IDS) == 5, f'expected 5 presentation types, got {len(PILOT_VIGNETTE_IDS)}: {sorted(by_type)}'

PILOT_ENTRIES = [e for e in all_entries if e['vignette_id'] in PILOT_VIGNETTE_IDS]
assert len(PILOT_ENTRIES) == 20, f'expected 20 pilot prompts, got {len(PILOT_ENTRIES)}'

## 7. vLLM helpers

`EPFLiGHT/Meditron3-8B` has no pre-quantized AWQ checkpoint on the Hub (checked 2026-08-06 -- searched, none found), so it loads via on-the-fly bitsandbytes NF4 quantization instead of a pre-quantized AWQ repo; the other three load pre-quantized AWQ checkpoints directly.

**T4-specific settings, verified by web search when this notebook was built (not guessed):**

- `dtype='float16'`, not `bfloat16`, for every model including the bitsandbytes path. T4 is Turing (compute capability 7.5) and lacks native bf16 tensor-core support -- several bitsandbytes/vLLM usage examples online default to `bfloat16` because they're written against Ampere+ GPUs; using that default here would silently run at a fraction of the expected speed, or fail outright, not obviously error with a clear message pointing at the cause.
- `quantization='awq'`, deliberately not letting vLLM auto-select an AWQ+Marlin kernel path -- Marlin is tuned for/requires Ampere (SM80) or newer, and there are open vLLM GitHub issues (#3392, #6985) about AWQ+Marlin conflicts. Plain AWQ (no Marlin) is the broadly-compatible path on Turing.
- `gpu_memory_utilization=0.85` -- within the community-reported safe range for an 8B AWQ/NF4 model on a 16GB T4 (reports cite up to 0.92-0.95 working reliably); kept at the more conservative end of that range for this notebook's *first* real run specifically, since actual utilization has been reported to overshoot the requested value on Colab T4s in practice. Raise it after you've confirmed a full run completes without OOM, if you want more KV-cache headroom.
- `max_model_len=4096` covers the longest real Arm 4 prompt (~2500 tok, see `data/prompts/manifest.json`'s `estimated_tokens`) plus `GENERATION_MAX_TOKENS` output, with margin.
- `tokenizer_revision` is pinned to the same commit as `revision` explicitly -- vLLM treats them as independent parameters, so leaving `tokenizer_revision` unset would default to `main` even with the model weights pinned, which is exactly the silent-mid-run-change risk pinning `revision` was meant to close.

In [ ]:
import gc
import time

import torch
from vllm import LLM, EngineArgs, LLMEngine, SamplingParams

GENERATION_MAX_TOKENS = 1600  # matches config/models.yaml generation.max_tokens


def _common_kwargs(model_cfg: dict) -> dict:
    kwargs = dict(
        model=model_cfg['model'],
        revision=model_cfg['revision'],
        tokenizer_revision=model_cfg['revision'],  # see note above -- not implied by `revision`
        dtype='float16',  # NOT bfloat16 -- T4/Turing lacks native bf16 tensor-core support
        gpu_memory_utilization=0.85,
        max_model_len=4096,
    )
    if model_cfg['quantization'] == 'bitsandbytes-nf4':
        kwargs['quantization'] = 'bitsandbytes'
        kwargs['load_format'] = 'bitsandbytes'
    else:
        kwargs['quantization'] = 'awq'  # not 'awq_marlin' -- see T4/Marlin note above
    return kwargs


def load_vllm_model(model_cfg: dict) -> LLM:
    """Pilot-only: the simple synchronous LLM() wrapper. Fine for the pilot,
    which just needs a single batched call and doesn't need per-response
    granularity -- the full run below uses the lower-level LLMEngine instead,
    for reasons explained in Section 10.
    """
    return LLM(**_common_kwargs(model_cfg))


def unload_vllm_model(llm) -> None:
    del llm
    gc.collect()
    torch.cuda.empty_cache()


def read_prompt_text(entry: dict) -> str:
    rel = entry['path'].removeprefix('data/prompts/')
    return (PROMPTS_DIR / rel).read_text(encoding='utf-8')


def generate_batch(llm: LLM, entries: list[dict], temperature: float = 0.0):
    sampling_params = SamplingParams(
        temperature=temperature, top_p=1.0, max_tokens=GENERATION_MAX_TOKENS,
    )
    prompts = [read_prompt_text(e) for e in entries]
    start = time.time()
    outputs = llm.generate(prompts, sampling_params)
    elapsed = time.time() - start
    return outputs, elapsed


def save_pilot_results(family: str, entries: list[dict], texts: list[str],
                        output_token_counts: list[int], parse_results, elapsed: float) -> Path:
    """Written to Drive immediately after each pilot run -- a disconnect right
    after the pilot finishes (before you've read the printed report) must not
    force re-running 20 GPU generations just to see the numbers again.
    """
    payload = {
        'family': family,
        'elapsed_seconds': elapsed,
        'responses': [
            {
                'vignette_id': e['vignette_id'],
                'arm': e['arm'],
                'output_tokens': out_tok,
                'parseable': r.parseable,
                'parseable_reason': r.reason,
                'text': text,
            }
            for e, text, out_tok, r in zip(entries, texts, output_token_counts, parse_results)
        ],
    }
    out_path = PILOT_RESULTS_DIR / f'{family}.json'
    out_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    print(f'Pilot results saved to {out_path}')
    return out_path

## 8. TASK A -- Pilot: Meta-Llama-3.1-8B-Instruct

Real measured numbers, not the planning-time 700-token assumption.

In [ ]:
llama_cfg = next(m for m in MODEL_REGISTRY if m['family'] == 'meta')
print(f"=== PILOT: {llama_cfg['model']} ===")

llm = load_vllm_model(llama_cfg)
outputs, elapsed = generate_batch(llm, PILOT_ENTRIES)

output_token_counts = [len(o.outputs[0].token_ids) for o in outputs]
texts = [o.outputs[0].text for o in outputs]

mean_out = sum(output_token_counts) / len(output_token_counts)
max_out = max(output_token_counts)
throughput = sum(output_token_counts) / elapsed

print(f'Mean output tokens/response: {mean_out:.1f}')
print(f'Max output tokens/response: {max_out}')
print(f'Wall time for 20 prompts: {elapsed:.1f}s')
print(f'Aggregate throughput: {throughput:.1f} tok/s (batched, T4)')

parse_results = [check_parseable(t) for t in texts]
n_parseable = sum(r.parseable for r in parse_results)
print(f'Parseable: {n_parseable}/{len(texts)}')
for e, r in zip(PILOT_ENTRIES, parse_results):
    if not r.parseable:
        print(f"  NOT PARSEABLE: {e['vignette_id']} arm{e['arm']}: {r.reason}")

llama_full_run_eta_min = (400 * mean_out / throughput) / 60
print(f'Corrected full-run (400 prompts) ETA for this model, using MEASURED throughput: {llama_full_run_eta_min:.1f} min')
print(f"(Planning-time estimate assumed 700 tok/response at ~43 tok/s single-stream = ~2.0h unbatched; "
      f"compare against the measured number above, not the planning assumption.)")

save_pilot_results('meta', PILOT_ENTRIES, texts, output_token_counts, parse_results, elapsed)
unload_vllm_model(llm)

## 9. TASK A -- Pilot: Meditron3-8B

Continued-pretrained models sometimes degrade at instruction-following relative to their base -- if this comes back with a much lower parseable rate than Llama's, that needs to be known now, not at hour six of the full run.

In [ ]:
meditron_cfg = next(m for m in MODEL_REGISTRY if m['family'] == 'epfl')
print(f"=== PILOT: {meditron_cfg['model']} ===")

llm = load_vllm_model(meditron_cfg)
outputs, elapsed = generate_batch(llm, PILOT_ENTRIES)

meditron_output_token_counts = [len(o.outputs[0].token_ids) for o in outputs]
meditron_texts = [o.outputs[0].text for o in outputs]

meditron_mean_out = sum(meditron_output_token_counts) / len(meditron_output_token_counts)
meditron_throughput = sum(meditron_output_token_counts) / elapsed

print(f'Mean output tokens/response: {meditron_mean_out:.1f}')
print(f'Max output tokens/response: {max(meditron_output_token_counts)}')
print(f'Wall time for 20 prompts: {elapsed:.1f}s')
print(f'Aggregate throughput: {meditron_throughput:.1f} tok/s (batched, T4)')

meditron_parse_results = [check_parseable(t) for t in meditron_texts]
meditron_n_parseable = sum(r.parseable for r in meditron_parse_results)
print(f'Parseable: {meditron_n_parseable}/{len(meditron_texts)}')
for e, r in zip(PILOT_ENTRIES, meditron_parse_results):
    if not r.parseable:
        print(f"  NOT PARSEABLE: {e['vignette_id']} arm{e['arm']}: {r.reason}")

if meditron_n_parseable < n_parseable:
    print()
    print(f'*** WARNING: Meditron3-8B parseable rate ({meditron_n_parseable}/20) is LOWER than '
          f'Llama-3.1-8B-Instruct ({n_parseable}/20). This is the exact instruction-following '
          f'degradation risk flagged before running -- inspect meditron_texts by hand before '
          f'committing to the full run for this model. ***')

meditron_full_run_eta_min = (400 * meditron_mean_out / meditron_throughput) / 60
print(f'Corrected full-run ETA for this model: {meditron_full_run_eta_min:.1f} min')

save_pilot_results('epfl', PILOT_ENTRIES, meditron_texts, meditron_output_token_counts,
                    meditron_parse_results, elapsed)
unload_vllm_model(llm)

## STOP -- read the pilot report above before continuing

Confirm before proceeding to the full run (Task B):

- Mean/max output tokens are in a sane range (not near `GENERATION_MAX_TOKENS`, which would mean responses are being truncated).
- Parseable rate is high (ideally 20/20) for both models. If Meditron's is meaningfully lower, read `meditron_texts` by hand -- it may need a different parsing strategy downstream, not a rerun.
- The corrected ETA (measured throughput, not the planning-time 700-token guess) times 4 models fits your session-time budget.

Once satisfied, set `PILOT_APPROVED = True` in the next cell to unlock the full run -- this is a deliberate gate, not a formality.

In [ ]:
PILOT_APPROVED = False  # <-- change to True only after reading the pilot report above

## 10. TASK B -- Full run: all 4 models x 400 prompts x 1 seed

Single seed at temperature 0 (see `results/PREREGISTRATION.md` Amendments, open-weight pivot -- zero compute budget is the reason, not a methodological preference). Checkpoints after every response; resumable; identical JSON shape to the API-path cache; one RULE 1 manifest per model.

**Why `LLMEngine` here and not the simpler `LLM.generate()` the pilot used:** `LLM.generate(prompts, ...)` is synchronous -- it blocks until *every* prompt in the call is done and returns them all at once. Calling it once per prompt in a loop (the obvious way to get per-response checkpointing) would process prompts one at a time and forfeit vLLM's whole continuous-batching throughput advantage -- the pilot's measured throughput, which came from a single batched call, would then systematically overstate the full run's actual speed. `LLMEngine.add_request()` + `.step()` is vLLM's lower-level streaming API: every pending prompt is added up front, the engine batches and decodes them together exactly as `LLM.generate()` does internally, and each response becomes available -- and gets checkpointed -- the moment *that* request finishes, independent of the others still running. This gets genuine continuous batching and true per-response checkpointing at the same time.

In [ ]:
assert PILOT_APPROVED, 'Set PILOT_APPROVED = True in the cell above after reading the pilot report.'

SEED = 0
TEMPERATURE = 0.0
PROGRESS_EVERY = 25  # 400 prompts/model -- more frequent than the API harness's 500, since a model-local run is much shorter

In [ ]:
def load_vllm_engine(model_cfg: dict) -> LLMEngine:
    engine_args = EngineArgs(**_common_kwargs(model_cfg))
    return LLMEngine.from_engine_args(engine_args)


def unload_vllm_engine(engine: LLMEngine) -> None:
    del engine
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
import hashlib
import subprocess
from datetime import UTC, datetime


def repo_git_commit_sha() -> str:
    try:
        return subprocess.check_output(
            ['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True
        ).strip()
    except Exception:
        return 'unknown'


def prompts_manifest_hash() -> str:
    return hashlib.sha256((PROMPTS_DIR / 'manifest.json').read_bytes()).hexdigest()


def run_model_full(model_cfg: dict) -> None:
    family = model_cfg['family']
    print(f"\n=== FULL RUN: {family}/{model_cfg['model']} ===")

    all_keys = [
        cache_key_for(
            family=family,
            model=model_cfg['model'],
            system_prompt='',
            messages=[{'role': 'user', 'content': read_prompt_text(e)}],
            params=GenerationParams(
                temperature=TEMPERATURE, top_p=1.0, max_tokens=GENERATION_MAX_TOKENS, seed=SEED
            ),
        )
        for e in all_entries
    ]
    key_to_entry = dict(zip(all_keys, all_entries))

    todo = pending_keys(all_keys, RESPONSES_DIR)
    already_done = len(all_keys) - len(todo)
    print(f'{already_done}/{len(all_keys)} already checkpointed (resuming)' if already_done
          else f'0/{len(all_keys)} checkpointed -- starting fresh')
    if not todo:
        print('Nothing to do for this model.')
        return

    engine = load_vllm_engine(model_cfg)
    total_input_tokens = 0
    total_output_tokens = 0
    start = time.time()

    sampling_params = SamplingParams(
        temperature=TEMPERATURE, top_p=1.0, max_tokens=GENERATION_MAX_TOKENS,
    )

    # Add every pending prompt up front -- vLLM's own scheduler (bounded by
    # gpu_memory_utilization's KV-cache budget) decides how many actually run
    # concurrently; requests that don't fit yet simply wait, no manual
    # windowing needed here.
    in_flight: dict[str, dict] = {}
    for key in todo:
        entry = key_to_entry[key]
        engine.add_request(key, read_prompt_text(entry), sampling_params)
        in_flight[key] = entry

    n_done = 0
    while in_flight:
        for output in engine.step():
            if not output.finished:
                continue
            key = output.request_id
            entry = in_flight.pop(key)

            response_text = output.outputs[0].text
            in_tok = len(output.prompt_token_ids)
            out_tok = len(output.outputs[0].token_ids)
            total_input_tokens += in_tok
            total_output_tokens += out_tok

            write_response_atomic(
                RESPONSES_DIR,
                key,
                text=response_text,
                raw={
                    'vignette_id': entry['vignette_id'],
                    'arm': entry['arm'],
                    'model': model_cfg['model'],
                    'model_revision': model_cfg['revision'],
                    'quantization': model_cfg['quantization'],
                    'seed': SEED,
                    'finish_reason': output.outputs[0].finish_reason,
                },
                input_tokens=in_tok,
                output_tokens=out_tok,
            )

            n_done += 1
            if n_done % PROGRESS_EVERY == 0 or n_done == len(todo):
                elapsed = time.time() - start
                rate = n_done / elapsed if elapsed > 0 else 0
                remaining = (len(todo) - n_done) / rate if rate > 0 else float('inf')
                print(f'  {n_done}/{len(todo)} done, {elapsed/60:.1f}min elapsed, '
                      f'ETA {remaining/60:.1f}min remaining, {len(in_flight)} still in flight')

    unload_vllm_engine(engine)

    # RULE 1 manifest -- one per model, since a Colab run is naturally sequential
    # per model (VRAM constraints preclude loading more than one 8B model at once).
    run_id = f"open-weight-{family}-seed{SEED}-{datetime.now(UTC).strftime('%Y%m%dT%H%M%SZ')}"
    manifest = {
        'run_id': run_id,
        'model_identifiers': [{
            'family': family,
            'name': model_cfg['model'],
            'version': model_cfg['revision'],
        }],
        'temperature': TEMPERATURE,
        'top_p': 1.0,
        'max_tokens': GENERATION_MAX_TOKENS,
        'seed': SEED,
        'prompt_template_hash': prompts_manifest_hash(),
        'vignette_set_version': 'v1',
        'git_commit_sha': repo_git_commit_sha(),
        'utc_timestamp': datetime.now(UTC).isoformat(),
        'total_input_tokens': total_input_tokens,
        'total_output_tokens': total_output_tokens,
        'quantization': model_cfg['quantization'],
        'base_model': model_cfg['base_model'],
        'base_model_revision': model_cfg['base_model_revision'],
        'n_responses_this_run': len(todo),
        'n_responses_total': len(all_keys),
    }
    (MANIFEST_DIR / f'{run_id}.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print(f'Manifest written: {run_id}.json')

In [ ]:
for model_cfg in MODEL_REGISTRY:
    run_model_full(model_cfg)

print('\nAll models done (or resumed to completion).')

## 11. Post-run sanity check

Confirms every (model, prompt) pair produced exactly one checkpointed response, before you copy `data/responses/` and `results/manifests/` back into the main repo.

In [ ]:
expected = len(MODEL_REGISTRY) * len(all_entries)  # models x 400, single seed
actual = len(load_completed_keys(RESPONSES_DIR))
print(f'Expected {expected} responses (4 models x 400 prompts x 1 seed), found {actual} checkpointed.')

empty_or_truncated = []
for key in load_completed_keys(RESPONSES_DIR):
    data = json.loads(checkpoint_path(RESPONSES_DIR, key).read_text(encoding='utf-8'))
    if not data['text'].strip():
        empty_or_truncated.append((key, 'empty'))
    elif data['raw'].get('finish_reason') == 'length':
        empty_or_truncated.append((key, 'truncated (hit max_tokens)'))

print(f'Empty or truncated responses: {len(empty_or_truncated)}')
for key, reason in empty_or_truncated[:20]:
    print(f'  {key}: {reason}')